# Classic Algorithms From Scratch

Companion notebook for the [Algorithms from Scratch lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/03-algorithms-from-scratch).

**The idea in one sentence.** Interview-classic ML algorithms — **linear regression** (normal
equation vs gradient descent), **K-Means**, and **decision-tree splits** — are short to
implement from scratch, and doing so exposes exactly where each one breaks in practice.

What we build and verify:

- **Linear regression:** the closed-form normal equation and gradient descent **converge to
  the same coefficients**.
- **K-Means:** Lloyd's algorithm, whose **inertia never increases** — but which can land in a
  bad local optimum.
- **Decision trees:** **Gini impurity** and the best-split search that finds the perfect cut.

We run all three, **validate convergence, monotone inertia, and the optimal split**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Linear Regression: Normal Equation vs Gradient Descent

In [ ]:
class LinearRegressionNE:
    """Linear regression via normal equation: θ = (X'X)^{-1} X'y."""
    
    def fit(self, X, y, ridge_lambda=1e-6):
        # Add bias column
        Xb = np.column_stack([np.ones(len(X)), X])
        # Ridge: (X'X + λI)^{-1} X'y for numerical stability
        I = np.eye(Xb.shape[1])
        I[0, 0] = 0  # don't regularize bias
        self.theta = np.linalg.solve(Xb.T @ Xb + ridge_lambda * I, Xb.T @ y)
        return self
    
    def predict(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return Xb @ self.theta

class LinearRegressionGD:
    """Linear regression via gradient descent."""
    
    def fit(self, X, y, lr=0.01, n_epochs=1000):
        Xb = np.column_stack([np.ones(len(X)), X])
        self.theta = np.zeros(Xb.shape[1])
        m = len(y)
        self.losses = []
        for _ in range(n_epochs):
            grad = Xb.T @ (Xb @ self.theta - y) / m
            self.theta -= lr * grad
            self.losses.append(np.mean((Xb @ self.theta - y)**2))
        return self
    
    def predict(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return Xb @ self.theta

# Generate data
X = np.random.randn(100, 2)
y = 3 * X[:, 0] - 2 * X[:, 1] + 1 + np.random.randn(100) * 0.5

ne = LinearRegressionNE().fit(X, y)
gd = LinearRegressionGD().fit(X, y)

print("True coefficients: [bias=1, x1=3, x2=-2]")
print(f"Normal equation:   {ne.theta.round(3)}")
print(f"Gradient descent:  {gd.theta.round(3)}")

### Validate: normal equation and gradient descent find the same solution

Both methods minimise the same least-squares objective, so they should recover the true
coefficients $[\text{bias}=1,\ x_1=3,\ x_2=-2]$ and agree with each other. We confirm both.

In [ ]:
true = np.array([1.0, 3.0, -2.0])
print(f'true            : {true}')
print(f'normal equation : {ne.theta.round(3)}')
print(f'gradient descent: {gd.theta.round(3)}')
assert np.allclose(ne.theta, true, atol=0.3), 'normal equation recovers the true coefficients'
assert np.allclose(gd.theta, true, atol=0.3), 'gradient descent recovers the true coefficients'
assert np.allclose(ne.theta, gd.theta, atol=0.1), 'both methods converge to the same least-squares solution'
print('\n✅ closed-form and iterative optimisation land on the same minimum of a convex loss')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(gd.losses, color='#6366f1', linewidth=2)
ax.axhline(np.mean((ne.predict(X) - y)**2), color='#2dd4bf', linestyle='--',
           label=f'Normal equation MSE (optimal)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Gradient descent converges to normal equation solution', fontsize=11)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
plt.tight_layout()
plt.show()

## K-Means from scratch

In [ ]:
class KMeans:
    def __init__(self, k=3, max_iter=100, tol=1e-4):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
    
    def fit(self, X):
        # K-means++ initialization
        centroids = [X[np.random.randint(len(X))]]
        for _ in range(self.k - 1):
            dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
            probs = dists / dists.sum()
            centroids.append(X[np.random.choice(len(X), p=probs)])
        self.centroids = np.array(centroids)
        
        self.inertias = []
        for _ in range(self.max_iter):
            # Assign
            dists = np.array([np.sum((X - c)**2, axis=1) for c in self.centroids])
            labels = np.argmin(dists, axis=0)
            # Update
            new_centroids = np.array([X[labels == k].mean(0) if (labels == k).any()
                                      else self.centroids[k] for k in range(self.k)])
            inertia = sum(np.sum((X[labels == k] - new_centroids[k])**2)
                         for k in range(self.k) if (labels == k).any())
            self.inertias.append(inertia)
            if np.max(np.abs(new_centroids - self.centroids)) < self.tol:
                break
            self.centroids = new_centroids
        self.labels_ = labels
        return self

# Generate 3-cluster data
centers = [[-3, -3], [0, 3], [3, -2]]
X_km = np.vstack([np.random.randn(100, 2) + c for c in centers])
km = KMeans(k=3).fit(X_km)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#6366f1', '#2dd4bf', '#f97316']

axes[0].scatter(X_km[:, 0], X_km[:, 1], c=[colors[l] for l in km.labels_], alpha=0.6, s=20)
axes[0].scatter(km.centroids[:, 0], km.centroids[:, 1], c='white', s=150, zorder=5, marker='*')
axes[0].set_title('K-Means clustering (K=3)', fontsize=11)

axes[1].plot(km.inertias, 'o-', color='#6366f1', linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Inertia (within-cluster variance)')
axes[1].set_title('K-Means convergence', fontsize=11)

plt.tight_layout()
plt.show()

### Validate: K-Means inertia never increases (Lloyd's algorithm descends)

Each K-Means iteration (assign, then update) can only *lower* the within-cluster sum of
squares — the objective is monotonically non-increasing, which is why the algorithm
converges. We confirm the inertia curve never goes up, and that the recovered centroids sit
near the true cluster centers.

In [ ]:
ins = km.inertias
print(f'inertia: {[round(v, 1) for v in ins]}')
assert all(ins[i] >= ins[i+1] - 1e-6 for i in range(len(ins) - 1)), 'inertia never increases'
for c in centers:
    nearest = min(np.sum((km.centroids - np.array(c))**2, axis=1))
    assert nearest < 1.0, 'each true center has a recovered centroid near it'
print('\n✅ Lloyd’s algorithm monotonically reduces inertia and recovers the three clusters')

## Decision tree: Gini impurity and best split

In [ ]:
def gini(y):
    """Gini impurity = 1 - sum(p_k^2)."""
    if len(y) == 0:
        return 0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs**2)

def information_gain(y, y_left, y_right):
    n = len(y)
    return gini(y) - (len(y_left)/n * gini(y_left) + len(y_right)/n * gini(y_right))

def best_split(X_col, y):
    """Find best threshold for a single feature column."""
    best_gain, best_thresh = -1, None
    thresholds = np.unique(X_col)
    for t in thresholds[:-1]:
        left = y[X_col <= t]
        right = y[X_col > t]
        gain = information_gain(y, left, right)
        if gain > best_gain:
            best_gain = gain
            best_thresh = t
    return best_thresh, best_gain

# Demo
X_tree = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y_tree = np.array([0, 0, 0, 0, 1, 1, 1, 1])

thresh, gain = best_split(X_tree, y_tree)
print(f"Best split threshold: {thresh}, information gain: {gain:.4f}")
print(f"Root Gini: {gini(y_tree):.4f}")
print(f"After split: left Gini={gini(y_tree[X_tree<=thresh]):.4f}, right Gini={gini(y_tree[X_tree>thresh]):.4f}")

### Validate: the best split finds the perfect cut

The toy data is perfectly separable at $x = 4$ (labels flip from 0 to 1). The best-split
search should find that threshold, and the information gain should equal the full root Gini
($0.5$) because both children become pure. We confirm.

In [ ]:
print(f'best threshold: {thresh}, gain: {gain:.4f}, root Gini: {gini(y_tree):.4f}')
assert thresh == 4.0, 'the optimal threshold separates the two classes'
assert gini(y_tree[X_tree <= thresh]) == 0 and gini(y_tree[X_tree > thresh]) == 0, 'both children are pure'
assert abs(gain - gini(y_tree)) < 1e-9, 'a perfect split recovers the entire root impurity as gain'
print('\n✅ Gini-based best-split finds the cut that maximally purifies the children')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **normal equation cost** | $(X^\top X)^{-1}$ is $O(d^3)$ and fails if $X^\top X$ is singular — ridge/GD help |
| **GD learning rate** | too large diverges, too small crawls; tune it |
| **K-Means local optima** | non-convex; a single init can be poor (demo) — use n_init |
| **choosing K** | inertia always falls with K; use the elbow / silhouette |
| **greedy tree splits** | locally-best splits over-fit; prune or limit depth |

Demo: K-Means lands in different local optima across random inits.

In [ ]:
# K-Means' biggest gotcha: it minimises a NON-convex objective, so different random
# initialisations can land in different local optima. On the 3 well-separated blobs above,
# k-means++ finds the global optimum every time (a feature!). The failure shows up when you
# ask for MORE clusters than exist (k=5 on 3 blobs): now the split is ambiguous and inits
# disagree. That is why real implementations run n_init times and keep the lowest inertia.
finals = []
for s in range(25):
    np.random.seed(s)
    finals.append(KMeans(k=5).fit(X_km).inertias[-1])   # k=5 on data with 3 true clusters
finals = np.array(finals)
n_bad = int((finals > finals.min() * 1.05).sum())
print(f'final inertia over 25 inits (k=5 on 3 blobs): best {finals.min():.1f}, worst {finals.max():.1f}')
print(f'{n_bad} of 25 runs landed >5% above the best -> local optima')
assert finals.max() > finals.min() + 1e-6 and n_bad > 0, 'different inits reach different local optima'
print('\nK-Means is not convex -> run it several times (n_init) and keep the best; a single run can be poor.')

## ✏️ Your turn

### Exercise 1: Implement ridge regression

Add L2 regularization to the normal equation: θ = (XᵀX + λI)⁻¹Xᵀy.

In [ ]:
def ridge_regression(X, y, lambda_reg):
    """
    Fit ridge regression using the regularized normal equation.
    
    Args:
        X: np.ndarray (n, p) — feature matrix (WITHOUT bias column)
        y: np.ndarray (n,) — target
        lambda_reg: float — L2 regularization strength
    Returns:
        np.ndarray: weight vector of shape (p+1,) — [bias, w1, w2, ...]
    """
    # TODO(you): add bias column, form the regularized normal equation
    # Don't regularize the bias term (identity row/col for features only)
    pass


# Test
theta_no_reg = ridge_regression(X, y, lambda_reg=0)
theta_reg = ridge_regression(X, y, lambda_reg=10)
print(f"No regularization:  {theta_no_reg.round(3)}")
print(f"With regularization: {theta_reg.round(3)}")
print(f"True values:         [1.0, 3.0, -2.0]")

In [ ]:
theta_no_reg = ridge_regression(X, y, 0)
theta_reg = ridge_regression(X, y, 100)
assert theta_no_reg is not None, "Should return weights"
assert theta_no_reg.shape == (3,), f"Expected shape (3,), got {theta_no_reg.shape}"
# High regularization should shrink feature weights toward 0
assert np.abs(theta_reg[1:]).max() < np.abs(theta_no_reg[1:]).max(), \
    "Stronger regularization should shrink feature weights"

# Edge case: a single-row dataset. XtX is rank-1 (singular) with lambda_reg=0,
# so a solve would blow up -- regularization is what makes this well-posed.
X_single = np.array([[1.0, 2.0]])
y_single = np.array([5.0])
theta_single = ridge_regression(X_single, y_single, lambda_reg=10.0)
assert theta_single.shape == (3,), f"Expected shape (3,), got {theta_single.shape}"
assert np.all(np.isfinite(theta_single)), "Regularized solve on a single row should stay finite"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ridge_regression(X, y, lambda_reg):
    Xb = np.column_stack([np.ones(len(X)), X])
    p = Xb.shape[1]
    I = np.eye(p)
    I[0, 0] = 0  # don't regularize bias
    return np.linalg.solve(Xb.T @ Xb + lambda_reg * I, Xb.T @ y)
```
</details>

### Exercise 2: Implement Gini impurity for a split

Given a feature column and labels, compute the weighted Gini impurity after splitting at a given threshold.

In [ ]:
def gini_after_split(feature_col, labels, threshold):
    """
    Compute the weighted Gini impurity of a split.

    Weighted Gini = (n_left/n) * Gini(left) + (n_right/n) * Gini(right)

    Args:
        feature_col: np.ndarray (n,) — feature values
        labels: np.ndarray (n,) — class labels
        threshold: float — split point (left: ≤ threshold, right: > threshold)
    Returns:
        float: weighted Gini impurity, lower is better. Return 0.0 for an
        empty dataset (n == 0) rather than dividing by zero.
    """
    # TODO(you): guard the empty-dataset case (n == 0) first, then split labels
    # into left and right using threshold and return the weighted average of
    # Gini impurities
    pass


feat = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
labs = np.array([0, 0, 0, 1, 1, 1])
print(f"Split at 3: {gini_after_split(feat, labs, 3):.4f} (should be near 0 — perfect split)")
print(f"Split at 1: {gini_after_split(feat, labs, 1):.4f} (poor split — all remaining are mixed)")

In [ ]:
feat = np.array([1., 2., 3., 4., 5., 6.])
labs = np.array([0, 0, 0, 1, 1, 1])
g_perfect = gini_after_split(feat, labs, 3)
g_poor = gini_after_split(feat, labs, 1)
assert g_perfect is not None, "Should return a float"
assert g_perfect < g_poor, "Perfect split should have lower weighted Gini than poor split"
assert abs(g_perfect) < 0.01, f"Perfect split (3/3 homogeneous groups) should give ~0 Gini, got {g_perfect}"

# Edge case: threshold exactly at the feature's minimum -> everything falls in
# the left group (<=), right group is empty.
g_at_min = gini_after_split(feat, labs, 1.0)
assert abs(g_at_min - 0.4) < 1e-9, f"Expected weighted Gini 0.4 at threshold=min, got {g_at_min}"

# Edge case: threshold exactly at the feature's maximum -> everything falls in
# the left group too (<=), so this should equal the *unsplit* root Gini.
g_at_max = gini_after_split(feat, labs, 6.0)
assert abs(g_at_max - gini(labs)) < 1e-9, "Threshold at the max should reduce to the root Gini"

# Edge case: single-row dataset -- one sample can't be split into two non-trivial
# groups, so the weighted Gini should be exactly 0.
g_single = gini_after_split(np.array([5.0]), np.array([1]), 5.0)
assert abs(g_single) < 1e-9, f"Single-row split should give 0 impurity, got {g_single}"

# Edge case: empty dataset -- must not divide by zero.
g_empty = gini_after_split(np.array([]), np.array([]), 3.0)
assert g_empty == 0.0, f"Empty dataset should give 0.0, got {g_empty}"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gini_after_split(feature_col, labels, threshold):
    n = len(labels)
    if n == 0:
        return 0.0
    left_mask = feature_col <= threshold
    right_mask = ~left_mask
    return (left_mask.sum() / n) * gini(labels[left_mask]) + \
           (right_mask.sum() / n) * gini(labels[right_mask])
```
</details>

### Exercise 3: dataset utility functions (DML 29, 30, 31, 33)

Four small utilities show up over and over as building blocks around the
algorithms above — shuffling, batching, threshold-based splitting, and
bootstrap-style subsetting. Implement all four:

- **DML 29 — `shuffle_data(X, y, seed=None)`**: randomly permute the rows of
  `X` and `y` together (the same permutation for both), reproducible via `seed`.
- **DML 30 — `batch_iterator(X, y=None, batch_size=64)`**: split `X` (and `y`,
  if given) into a list of consecutive batches of at most `batch_size` rows
  each (the last batch may be smaller).
- **DML 31 — `divide_on_feature(X, feature_i, threshold)`**: split the rows of
  `X` into `[X_at_or_above, X_below]` based on whether column `feature_i` is
  `>= threshold`.
- **DML 33 — `get_random_subsets(X, y, n_subsets, replacements=True, seed=42)`**:
  return `n_subsets` random `(X_subset, y_subset)` pairs — bootstrap-sized
  (`n` rows, sampled *with* replacement) if `replacements=True`, half-sized
  (`n // 2` rows, sampled *without* replacement) otherwise.

In [ ]:
def shuffle_data(X, y, seed=None):
    """Randomly permute the rows of X and y together.

    Args:
        X: np.ndarray (n, p)
        y: np.ndarray (n,)
        seed: optional int for reproducibility
    Returns:
        (X_shuffled, y_shuffled), same shapes as the inputs
    """
    # TODO(you): build a permutation of range(len(X)) with a seeded RNG
    # (np.random.RandomState(seed).shuffle works well), then index both X and y
    pass


def batch_iterator(X, y=None, batch_size=64):
    """Split X (and y, if given) into consecutive batches of at most batch_size rows.

    Returns:
        list of [X_batch, y_batch] pairs if y is not None, else a list of X_batch
    """
    # TODO(you): loop i = 0, batch_size, 2*batch_size, ... and slice X[i:i+batch_size]
    # (and y[i:i+batch_size] if y is not None)
    pass


def divide_on_feature(X, feature_i, threshold):
    """Split the rows of X on whether column feature_i is >= threshold.

    Returns:
        [X_at_or_above_threshold, X_below_threshold]
    """
    # TODO(you): boolean-mask rows where X[:, feature_i] >= threshold
    pass


def get_random_subsets(X, y, n_subsets, replacements=True, seed=42):
    """Return n_subsets random (X_subset, y_subset) pairs.

    subsample_size is len(X) (bootstrap) if replacements else len(X) // 2.
    """
    # TODO(you): subsample_size = n if replacements else n // 2; draw n_subsets
    # index arrays with a seeded RNG (np.random.RandomState(seed).choice), then
    # index X and y with each
    pass


# Smoke run
X_demo = np.array([[1., 2.], [3., 4.], [5., 6.], [7., 8.]])
y_demo = np.array([1, 2, 3, 4])
print("shuffled:      ", shuffle_data(X_demo, y_demo, seed=0))
print("batches (bs=3):", batch_iterator(X_demo, y_demo, batch_size=3))
print("divided @ 5:   ", divide_on_feature(X_demo, feature_i=0, threshold=5))
print("random subsets:", get_random_subsets(X_demo, y_demo, n_subsets=2, replacements=True, seed=42))

In [ ]:
X4 = np.array([[1., 2.], [3., 4.], [5., 6.], [7., 8.]])
y4 = np.array([1, 2, 3, 4])

# --- shuffle_data ---
Xs, ys = shuffle_data(X4, y4, seed=0)
assert Xs.shape == X4.shape and ys.shape == y4.shape
assert set(ys.tolist()) == set(y4.tolist()), "shuffle should permute, not lose/duplicate rows"
for xi, yi in zip(Xs, ys):
    row_idx = np.where(y4 == yi)[0][0]
    assert np.array_equal(xi, X4[row_idx]), "X row must stay paired with its original y"
Xs2, ys2 = shuffle_data(X4, y4, seed=0)
assert np.array_equal(Xs, Xs2) and np.array_equal(ys, ys2), "same seed -> same permutation"
# Edge case: empty dataset
Xe, ye = shuffle_data(np.empty((0, 2)), np.empty((0,)), seed=0)
assert Xe.shape == (0, 2) and ye.shape == (0,)
# Edge case: single-row dataset
Xo, yo = shuffle_data(np.array([[9., 9.]]), np.array([9]), seed=0)
assert np.array_equal(Xo, [[9., 9.]]) and np.array_equal(yo, [9])

# --- batch_iterator ---
batches = batch_iterator(X4, y4, batch_size=3)
assert len(batches) == 2, "4 rows / batch_size 3 -> 2 batches (3 + 1)"
assert batches[0][0].shape[0] == 3 and batches[1][0].shape[0] == 1
assert np.array_equal(np.vstack([b[0] for b in batches]), X4)
batches_x_only = batch_iterator(X4, batch_size=2)
assert len(batches_x_only) == 2 and all(b.shape[0] == 2 for b in batches_x_only)
# Edge case: empty dataset
assert batch_iterator(np.empty((0, 2)), batch_size=3) == []
# Edge case: single-row dataset, batch_size larger than the dataset
single_batches = batch_iterator(np.array([[1., 1.]]), np.array([1]), batch_size=64)
assert len(single_batches) == 1 and single_batches[0][0].shape[0] == 1

# --- divide_on_feature ---
above, below = divide_on_feature(X4, feature_i=0, threshold=5)
assert np.array_equal(above, [[5., 6.], [7., 8.]])
assert np.array_equal(below, [[1., 2.], [3., 4.]])
# Edge case: threshold exactly at the feature's minimum -> everything is "above" (>=)
above_min, below_min = divide_on_feature(X4, feature_i=0, threshold=1)
assert len(below_min) == 0 and len(above_min) == 4
# Edge case: threshold exactly at the feature's maximum -> only the max row is "above"
above_max, below_max = divide_on_feature(X4, feature_i=0, threshold=7)
assert len(above_max) == 1 and len(below_max) == 3
# Edge case: empty dataset
above_e, below_e = divide_on_feature(np.empty((0, 2)), feature_i=0, threshold=5)
assert len(above_e) == 0 and len(below_e) == 0
# Edge case: single-row dataset
above_s, below_s = divide_on_feature(np.array([[10., 0.]]), feature_i=0, threshold=10)
assert len(above_s) == 1 and len(below_s) == 0

# --- get_random_subsets ---
subsets_boot = get_random_subsets(X4, y4, n_subsets=3, replacements=True, seed=42)
assert len(subsets_boot) == 3
for Xsub, ysub in subsets_boot:
    assert Xsub.shape[0] == len(X4), "bootstrap subsets should match dataset size"
subsets_noreplace = get_random_subsets(X4, y4, n_subsets=3, replacements=False, seed=42)
for Xsub, ysub in subsets_noreplace:
    assert Xsub.shape[0] == len(X4) // 2, "no-replacement subsets should be half-sized"
    assert len(set(map(tuple, Xsub.tolist()))) == Xsub.shape[0], "no-replacement subset shouldn't repeat rows"
# Edge case: single-row dataset
X1, y1 = np.array([[1., 1.]]), np.array([1])
subs_single = get_random_subsets(X1, y1, n_subsets=2, replacements=True, seed=0)
assert all(Xsub.shape[0] == 1 for Xsub, ysub in subs_single)
# Edge case: empty dataset
subs_empty = get_random_subsets(np.empty((0, 2)), np.empty((0,)), n_subsets=2, replacements=True, seed=0)
assert all(Xsub.shape[0] == 0 for Xsub, ysub in subs_empty)

print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def shuffle_data(X, y, seed=None):
    rng = np.random.RandomState(seed)
    idx = np.arange(X.shape[0])
    rng.shuffle(idx)
    return X[idx], y[idx]

def batch_iterator(X, y=None, batch_size=64):
    n_samples = X.shape[0]
    batches = []
    for i in range(0, n_samples, batch_size):
        end = min(i + batch_size, n_samples)
        if y is not None:
            batches.append([X[i:end], y[i:end]])
        else:
            batches.append(X[i:end])
    return batches

def divide_on_feature(X, feature_i, threshold):
    mask = X[:, feature_i] >= threshold
    return [X[mask], X[~mask]]

def get_random_subsets(X, y, n_subsets, replacements=True, seed=42):
    rng = np.random.RandomState(seed)
    n_samples = X.shape[0]
    subsample_size = n_samples if replacements else n_samples // 2
    subsets = []
    for _ in range(n_subsets):
        idx = rng.choice(n_samples, size=subsample_size, replace=replacements)
        subsets.append((X[idx], y[idx]))
    return subsets
```

`divide_on_feature` and `get_random_subsets` both rely on plain boolean/fancy
indexing, which is why they fall out of empty- and single-row datasets for
free: an empty boolean mask selects zero rows, and `rng.choice(n, size=0)` is
a valid (empty) draw rather than an error.
</details>

## Key takeaways

- **Two paths to least squares:** the normal equation (closed form, $O(d^3)$) and gradient
  descent (iterative, scales to large $d$) reach the same minimum (verified).
- **K-Means descends monotonically:** inertia never increases (verified) — but the objective
  is non-convex, so a single run can get stuck (demo); use `n_init`.
- **Tree splits are greedy Gini reduction:** the best split maximally purifies the children;
  a perfect cut recovers the whole root impurity as gain (verified).
- **Implementing from scratch reveals the failure modes** — singular $X^\top X$, learning-rate
  tuning, local optima, greedy over-fitting.